# Yoruba

In [2]:
import os
import sys

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from tqdm import tqdm
import IPython.display as ipd

# ── Fix: torchaudio 2.10 defaults to torchcodec which needs FFmpeg libs ───────
# Monkey-patch torchaudio.load to use soundfile (works for .wav files)
_original_torchaudio_load = torchaudio.load

def _torchaudio_load_sf(filepath, *args, **kwargs):
    data, samplerate = sf.read(filepath, dtype="float32")
    tensor = torch.from_numpy(data.T if data.ndim > 1 else data[None, :])
    return tensor, samplerate

torchaudio.load = _torchaudio_load_sf
print("Patched torchaudio.load → soundfile backend")

Patched torchaudio.load → soundfile backend


In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_PATH = "/home/mila/g/guzmand/scratch/datasets/bible-tts-resources/Yoruba"
WAV_DIR      = os.path.join(DATASET_PATH, "wav")

TEST_TSV     = os.path.join(DATASET_PATH, "validation.tsv")
TRAIN_TSV    = os.path.join(DATASET_PATH, "train.tsv")

# Checkpoint & vocab  (edit CKPT_PATH to switch models)
CKPT_PATH    = "ckpts/F5TTS_v1_Base_vocos_pinyin_open-bible-yoruba/model_last.pt"
VOCAB_FILE   = "data/open-bible-yoruba_pinyin/vocab.txt"
MODEL_CFG    = "src/f5_tts/configs/F5TTS_v1_Base_Open_Bible_Yoruba.yaml"

OUTPUT_DIR   = "synthesis_output/F5TTS_v1_Base_vocos_pinyin_open-bible-yoruba"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load test set ──────────────────────────────────────────────────────────────
test = pd.read_csv(TEST_TSV, sep="\t")
print(f"Validation samples: {len(test)}")
test.head()

Validation samples: 1286


,filename,text,testament,book,chapter,verse,duration_seconds
0,New-Testament-1-Corinthians-001-009.wav,"Ọlọ́run, nípasẹ̀ ẹni ti a pè yín sínú ìdàpọ̀ p...",New Testament,1 Corinthians,1,9,10.96
1,New-Testament-1-Corinthians-003-014.wav,"Bí iṣẹ́ tí ẹnikẹ́ni bá ṣe lórí rẹ̀ bá dúró, òu...",New Testament,1 Corinthians,3,14,5.41
2,New-Testament-1-Corinthians-003-019.wav,Nítorí ọgbọ́n ayé yìí jẹ́ òmùgọ̀ lọ́dọ̀ Ọlọ́ru...,New Testament,1 Corinthians,3,19,10.45
3,New-Testament-1-Corinthians-005-001.wav,Ìròyìn rẹ ń tàn kalẹ̀ pé ìwà àgbèrè wa láàrín ...,New Testament,1 Corinthians,5,1,12.25
4,New-Testament-1-Corinthians-007-001.wav,"Ní báyìí, nítorí àwọn ohun tí ẹ ṣe kọ̀wé: Ó dá...",New Testament,1 Corinthians,7,1,9.84


In [7]:
# ── Pick a reference audio from the training set ──────────────────────────────
# F5-TTS clones the speaker from a short reference clip + its transcript.
# We pick one training utterance (~7-9 s) as the universal reference for this
# single-speaker dataset. Change REF_INDEX to try a different prompt.

train = pd.read_csv(TRAIN_TSV, sep="\t")

# Pick a reference utterance with a "typical" duration (6-10 s)
candidates = train[(train["duration_seconds"] >= 6) & (train["duration_seconds"] <= 10)]
REF_INDEX = 0  # index into the filtered candidates; change to try others

ref_row = candidates.iloc[REF_INDEX]
REF_AUDIO = os.path.join(WAV_DIR, ref_row["filename"])
REF_TEXT  = ref_row["text"]

print(f"Reference audio : {REF_AUDIO}")
print(f"Reference text  : {REF_TEXT}")
print(f"Reference dur   : {ref_row['duration_seconds']:.2f}s")

ipd.display(ipd.Audio(REF_AUDIO))

Reference audio : /home/mila/g/guzmand/scratch/datasets/bible-tts-resources/Yoruba/wav/New-Testament-1-Corinthians-001-001.wav
Reference text  : Paulu, ẹni ti a pé láti jẹ́ aposteli Kristi Jesu nípa ìfẹ́ Ọlọ́run àti Sostene arákùnrin wa,
Reference dur   : 8.93s


In [9]:
# ── Load model & vocoder ──────────────────────────────────────────────────────
from hydra.utils import get_class
from omegaconf import OmegaConf

from f5_tts.infer.utils_infer import (
    load_model,
    load_vocoder,
    preprocess_ref_audio_text,
    infer_process,
)

# Load config
model_cfg = OmegaConf.load(MODEL_CFG)
model_cls = get_class(f"f5_tts.model.{model_cfg.model.backbone}")
model_arc = model_cfg.model.arch

# Vocoder
vocoder = load_vocoder(vocoder_name="vocos", is_local=False)

# TTS model
ema_model = load_model(
    model_cls,
    model_arc,
    CKPT_PATH,
    mel_spec_type="vocos",
    vocab_file=VOCAB_FILE,
)
print("Model loaded ✓")

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  data/open-bible-yoruba_pinyin/vocab.txt
token :  custom
model :  ckpts/F5TTS_v1_Base_vocos_pinyin_open-bible-yoruba/model_last.pt 

Model loaded ✓


In [10]:
# ── Preprocess reference audio ─────────────────────────────────────────────────
ref_audio, ref_text = preprocess_ref_audio_text(REF_AUDIO, REF_TEXT)
print(f"Preprocessed ref_audio: {ref_audio}")
print(f"Preprocessed ref_text : {ref_text}")

Converting audio...
Using custom reference text...

ref_text   Paulu, ẹni ti a pé láti jẹ́ aposteli Kristi Jesu nípa ìfẹ́ Ọlọ́run àti Sostene arákùnrin wa,. 
Preprocessed ref_audio: /tmp/tmpz6wvr2be.wav
Preprocessed ref_text : Paulu, ẹni ti a pé láti jẹ́ aposteli Kristi Jesu nípa ìfẹ́ Ọlọ́run àti Sostene arákùnrin wa,. 


In [ ]:
# ── Batch inference over the validation set ────────────────────────────────────
# For each row in the validation TSV, synthesize speech from the text column
# using the same reference audio. Outputs are saved as individual .wav files.

generated_files = []
errors = []

for idx, row in tqdm(test.iterrows(), total=len(test), desc="Synthesizing"):
    gen_text = row["text"]
    out_filename = row["filename"]  # keep original filename
    out_path = os.path.join(OUTPUT_DIR, out_filename)

    # Skip if already generated
    if os.path.exists(out_path):
        generated_files.append(out_path)
        continue

    try:
        audio_segment, final_sample_rate, spectrogram = infer_process(
            ref_audio,
            ref_text,
            gen_text,
            ema_model,
            vocoder,
            mel_spec_type="vocos",
        )
        sf.write(out_path, audio_segment, final_sample_rate)
        generated_files.append(out_path)
    except Exception as e:
        print(f"Error on {out_filename}: {e}")
        errors.append({"filename": out_filename, "error": str(e)})

print(f"\nDone! {len(generated_files)} files generated, {len(errors)} errors.")
print(f"Output directory: {OUTPUT_DIR}")

Synthesizing: 100%|██████████| 1286/1286 [00:00<00:00, 28890.29it/s]


Done! 1286 files generated, 0 errors.
Output directory: synthesis_output/F5TTS_v1_Base_vocos_pinyin_open-bible-yoruba


In [12]:
# ── Quick sanity check: listen to a sample ─────────────────────────────────────
import IPython.display as ipd

if generated_files:
    sample_path = generated_files[0]
    print(f"Playing: {sample_path}")
    ipd.display(ipd.Audio(sample_path))

Playing: synthesis_output/F5TTS_v1_Base_vocos_pinyin_open-bible-yoruba/New-Testament-1-Corinthians-001-009.wav
